# Anharmonic Phonons From NVT Molecular Dynamics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stfc/janus-core/blob/main/docs/source/tutorials/python/anharmonic_phonons.ipynb)

This tutorial shows how to generate standard [TDEP](https://tdep-developers.github.io/tdep/) input files through `post_process_kwargs` while running an NVT molecular dynamics simulation with `janus-core`, and then use them to compute anharmonic phonon properties with TDEP.

The workflow is:

1. Build a unit cell and supercell.
2. Run a small NVT molecular dynamics simulation with `janus-core`.
3. Automatically generate standard TDEP input files.
4. Run TDEP separately to extract force constants and compute anharmonic phonon properties.

> [!NOTE]
> The `janus-core` part of this tutorial can be run directly in Python, but the TDEP commands require a local environment where TDEP is installed.




In [ ]:
from pathlib import Path

from ase.build import bulk
from ase.io import write

from janus_core.calculations.md import NVT
from janus_core.calculations.single_point import SinglePoint

## Build The Unit Cell And Supercell

We first construct a rocksalt NaCl unit cell and expand it to a `2 x 2 x 2` supercell.

The unit-cell structure is used to generate `infile.ucposcar`, while the supercell structure is used both for the molecular dynamics simulation and to generate `infile.ssposcar`.



In [ ]:
unit_cell = bulk("NaCl", "rocksalt", a=5.63, cubic=True)
supercell = unit_cell * (2, 2, 2)

write("unit_cell.cif", unit_cell)
write("supercell.cif", supercell)

Path("unit_cell.cif"), Path("supercell.cif")

## Run NVT Molecular Dynamics And Generate TDEP Inputs

We now run a small NVT molecular dynamics simulation using the supercell structure.

TDEP input generation is enabled through `post_process_kwargs`. This automatically writes the following files into `tdep-inputs/` after the MD run finishes:

- `infile.ucposcar`
- `infile.ssposcar`
- `infile.positions`
- `infile.forces`
- `infile.stat`
- `infile.meta`


In [ ]:
single_point = SinglePoint(
    struct="supercell.cif",
    arch="mace_mp",
    model="small",
)

nvt = NVT(
    struct=single_point.struct,
    temp=300.0,
    steps=200,
    timestep=1.0,
    traj_every=20,
    stats_every=20,
    file_prefix="janus_results/NaCl-nvt-T300",
    post_process_kwargs={
        "tdep_compute": True,
        "tdep_unit_cell_file": "unit_cell.cif",
        "tdep_supercell_file": "supercell.cif",
        "tdep_output_dir": "janus_results/tdep-inputs",
    },
)

nvt.run()

## Check The Generated TDEP Input Files

After the molecular dynamics simulation completes, the TDEP input files should be available in the `tdep-inputs/` directory.


In [ ]:
required_files = [
    "infile.ucposcar",
    "infile.ssposcar",
    "infile.positions",
    "infile.forces",
    "infile.stat",
    "infile.meta",
]

for name in required_files:
    path = Path("janus_results/tdep-inputs") / name
    print(name, path.exists(), path)

## Run TDEP

After generating the standard TDEP input files, run the following commands in a local environment where [TDEP](https://tdep-developers.github.io/tdep/) is installed:

```bash
cd janus_results/tdep-inputs
extract_forceconstants -rc2 100
cp outfile.forceconstant infile.forceconstant
phonon_dispersion_relations --dos -p
gnuplot outfile.dispersion_relations.gnuplot_pdf



## Inspect The TDEP Outputs

After the TDEP commands finish, you should see output files such as:

- `outfile.forceconstant`
- `outfile.dispersion_relations`
- `outfile.dispersion_relations.pdf`
- `outfile.phonon_dos`
- `outfile.phonon_dos.hdf5`

These files can then be used to inspect the anharmonic phonon dispersion and density of states.
The phonon dispersion generated by TDEP is shown below.

<img src="../data/outfile.dispersion_relations.png" alt="TDEP phonon dispersion" width="400">



